In [13]:
import os
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

rawData = "../data/raw/dataset.csv"
processedDir = "../data/processed"
outPath = os.path.join(processedDir, "dataset_full_preprocessed.csv")

df = pd.read_csv(rawData)
df.shape

(13910, 131)

In [14]:
df = df.drop(columns=["id"])
print("Dropped column: id")

y = df["Class"]
X = df.drop(columns=["Class"])

X_numeric = X.select_dtypes(include=[np.number])
X_numeric.shape

Dropped column: id


(13910, 129)

In [15]:
nan_ratio = X_numeric.isna().sum().sum()
nan_ratio

np.int64(0)

In [16]:
LQ = 0.01
RQ = 0.99

X_clipped = X_numeric.copy()

for col in X_clipped.columns:
    leftQ = X_clipped[col].quantile(LQ)
    rightQ = X_clipped[col].quantile(RQ)
    X_clipped[col] = X_clipped[col].clip(leftQ, rightQ)

In [17]:
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X_numeric)

scaler = RobustScaler(quantile_range=(5, 95))
X_scaled = scaler.fit_transform(X_imputed)

In [19]:
processed_df = pd.DataFrame(
    X_scaled,
    columns=X_numeric.columns
)

if y is not None:
    processed_df["Class"] = y.values

processed_df.to_csv(outPath, index=False)
processed_df.shape

(13910, 130)